In [11]:
from transformers import AutoModelForMultipleChoice,AutoTokenizer,TrainingArguments,Trainer
import evaluate
from datasets import DatasetDict
import torch

In [5]:
from datasets import DatasetDict
ds = DatasetDict.load_from_disk("./C3/")
ds

DatasetDict({
    test: Dataset({
        features: ['id', 'context', 'question', 'choice', 'answer'],
        num_rows: 1625
    })
    train: Dataset({
        features: ['id', 'context', 'question', 'choice', 'answer'],
        num_rows: 11869
    })
    validation: Dataset({
        features: ['id', 'context', 'question', 'choice', 'answer'],
        num_rows: 3816
    })
})

In [6]:
ds.pop("test")

Dataset({
    features: ['id', 'context', 'question', 'choice', 'answer'],
    num_rows: 1625
})

In [12]:
model_name = "bert-base-chinese"
tokenizer = AutoTokenizer.from_pretrained(model_name,local_files_only = True)

In [7]:
ds['train']['choice'][2]

['会是昨天开的', '男的没有参加', '讨论得不热烈', '参加的人很少']

In [ ]:
def process_fn(dataset):
    context = []
    question_choice = []
    labels = []
    for idx in range(len(dataset['question'])):
        ctx = "\n".join(dataset['context'][idx])
        question = dataset['question'][idx]
        choices = dataset['choice'][idx]
        for choice in choices: 
            context.append(ctx)
            question_choice.append(question +" "+choice)
        if len(choices)<4:
            for _ in range(4-len(choices)):
                context.append(ctx)
                question_choice.append(question +" "+"无法确定")
        labels.append(choices.index(dataset["answer"][idx]))
    tokenized_set = tokenizer(context, question_choice, truncation= "only_first",
                              max_length=256, padding = "max_length", return_tensors = "pt")
    tokenized_set = {k:v.reshape(len(labels), 4, -1) for k,v in tokenized_set.items()}       
    tokenized_set["labels"] = torch.tensor(labels)
    return tokenized_set

In [ ]:
res = ds['train'].select(range(10)).map(process_fn,batched=True)

In [33]:
torch.tensor(res['input_ids']).shape

torch.Size([10, 4, 256])

In [34]:
tokenized_set = ds.map(process_fn,batched=True)

Map:   0%|          | 0/11869 [00:00<?, ? examples/s]

Map:   0%|          | 0/3816 [00:00<?, ? examples/s]

In [35]:
tokenized_set

DatasetDict({
    train: Dataset({
        features: ['id', 'context', 'question', 'choice', 'answer', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 11869
    })
    validation: Dataset({
        features: ['id', 'context', 'question', 'choice', 'answer', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 3816
    })
})

In [13]:
model = AutoModelForMultipleChoice.from_pretrained(model_name,local_files_only = True)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForMultipleChoice LOAD REPORT from: bert-base-chinese
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
import numpy as np
accuracy = evaluate.load('accuracy')
def eval_metric(pred):
    predictions, labels = pred
    predictions = np.argmax(predictions,axis=-1)
    return accuracy.compute(predictions=predictions,references=labels)

In [4]:
args = TrainingArguments(
    output_dir="./muliple_choice",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=1,
    logging_steps=50,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    fp16=True
)

In [ ]:
trainer = Trainer(
    model=model,
    args=args,
    tokenizer=tokenizer,
    train_dataset=tokenized_set["train"],
    eval_dataset=tokenized_set["validation"],
    compute_metrics=eval_metric
)

NameError: name 'model' is not defined

In [ ]:
trainer.train()

In [23]:
class Multichoice_pipeline:
    def __init__(self,model,tokenizer,):
        self.model = model 
        self.tokenizer = tokenizer 
    
    def process(self,context,question,choices):
        contexts = []
        question_choice = []
        for choice in choices: 
            contexts.append(context)
            question_choice.append(question +" "+choice)
        return tokenizer(contexts, question_choice, truncation= "only_first",
                              max_length=256,return_tensors = "pt")
    
    def pred(self,inputs):
        inputs = {k:v.unsqueeze(0)  for k,v in inputs.items()}
        return self.model(**inputs).logits
    
    def postprocess(self,logits,choices):
        return choices[torch.argmax(logits,dim=-1).item()]
    
    def __call__(self, context, question, choices):
        inputs = self.process(context, question, choices)
        logits = self.pred(inputs)
        output = self.postprocess(logits,choices)
        return output

In [24]:
pipe = Multichoice_pipeline(model, tokenizer)

In [26]:
pipe("小明在北京上班", "小明在哪里上班？", ["北京", "上海","河北", "海南"])

'北京'